## Comparing dataset projections (Part 2)

In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Slope

In [2]:
# ~30 m resolution
slope = ee.Image("projects/deforestation-495419/assets/panama_slopee").clip(panama_geom)

Map.addLayer(slope, {"min": 0, "max": 30}, "Slope")

In [3]:
slope.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:3857',
 'transform': [30.97431682912205,
  0,
  -9245686.6909565,
  0,
  -30.97431682912204,
  1085041.4884436138]}

### Soils

In [4]:
# ~260m resolution
pa = ee.Image("projects/deforestation-495419/assets/Soil_Org_C").clip(panama_geom)
pn = ee.Image("projects/deforestation-495419/assets/Soil_N").clip(panama_geom)
ps = ee.Image("projects/deforestation-495419/assets/sand").clip(panama_geom)
ph = ee.Image("projects/deforestation-495419/assets/pH").clip(panama_geom)

Map.addLayer(pa, {"min": 0, "max": 150}, "Soil Organic Carbon")
Map.addLayer(pn, {"min": 0, "max": 80}, "Nitrogen")
Map.addLayer(ps, {"min": 0, "max": 1000}, "Sand")
Map.addLayer(ph, {"min": 40, "max": 80}, "pH")

In [5]:
pa.projection().getInfo()
pn.projection().getInfo()
ps.projection().getInfo()
ph.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.0022607385079125844, 0, -83, 0, -0.002390438247011952, 10]}

### Distance to deforested areas

In [6]:
hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").clip(panama_geom)

lossyear = hansen.select("lossyear")

recent_loss = lossyear.gte(20).selfMask()

distance_loss = (
    recent_loss.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_loss")
    .clip(panama_geom)
)

# Map.addLayer(recent_loss, {"palette": ["red"]}, "Recent Loss")
Map.addLayer(distance_loss, {"min": 0, "max": 5000}, "Distance to Loss")

In [7]:
distance_loss.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.00025, 0, -180, 0, -0.00025, 80]}

### Precipitation (monthly)

In [ ]:
# ~5566m resolution
chirps = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_SAT")
    .filterDate("2018-05-01", "2018-05-31")
    .select("precipitation")
)

# precip = chirps.mean().clip(panama_geom) --> lowers resolution BIG TIME (~111,319m)

Map.addLayer(
    chirps,
    {
        "min": 1,
        "max": 17,
        "palette": ["#001137", "#0aab1e", "#e7eb05", "#2c7fb8", "#253494"]
    },
    "Precipitation"
)

In [28]:
chirps.first().projection().nominalScale().getInfo()

5565.974622603162

### Temperature (monthly)

In [29]:
# ~11,131m resolution
temp = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2023-01-01", "2023-02-01")
    .first()
    .clip(panama_geom)
)

Map.addLayer(
    temp,
    {
        "bands": ["temperature_2m"],
        "min": 250,
        "max": 320
    },
    "Temperature"
)

In [30]:
temp.projection().nominalScale().getInfo()

11131.949079327358

### Distance to hydrological basins

In [31]:
basins = ee.Image("projects/deforestation-495419/assets/hydrographic_basins").clip(panama_geom)

basins_int = basins.toInt()
neighbors = basins_int.focal_mode(1)

edges = basins_int.neq(neighbors).selfMask()

dist_basin = (
    edges.fastDistanceTransform(512).sqrt()
    .multiply(basins.projection().nominalScale())
    .clip(panama_geom)
)

# Map.addLayer(basins, {}, "Basins")
# Map.addLayer(edges, {"palette": ["red"]}, "Basin Edges")
Map.addLayer(dist_basin, {"min": 0, "max": 50000}, "Distance Basin")

In [32]:
dist_basin.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:3857',
 'transform': [101.30653541413305,
  0,
  -9246721.428728944,
  0,
  -101.3065354141332,
  1168336.6903672446]}

### Generate map

In [33]:
Map

Map(bottom=15906.0, center=[8.51583556120223, -80.11230468750001], controls=(WidgetControl(options=['position'…